<a href="https://colab.research.google.com/github/areeba-munir/global-space-mission-analysis/blob/main/notebooks/Space_Missions_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

<center><img src="https://i.imgur.com/9hLRsjZ.jpg" height=400></center>

This dataset was scraped from [nextspaceflight.com](https://nextspaceflight.com/launches/past/?page=1) and includes all the space missions since the beginning of Space Race between the USA and the Soviet Union in 1957!

### Install Package with Country Codes

In [95]:
%pip install iso3166

### Upgrade Plotly

In [96]:
%pip install --upgrade plotly

### Import Statements

In [97]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

# These might be helpful:
from iso3166 import countries
from datetime import datetime, timedelta

### Notebook Presentation

In [98]:
pd.options.display.float_format = '{:,.2f}'.format

### Load the Data

In [99]:
df_data = pd.read_csv('mission_launches.csv')
df_data.head()

,Unnamed: 0.1,Unnamed: 0,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status
0,0,0,SpaceX,"LC-39A, Kennedy Space Center, Florida, USA","Fri Aug 07, 2020 05:12 UTC",Falcon 9 Block 5 | Starlink V1 L9 & BlackSky,StatusActive,50.0,Success
1,1,1,CASC,"Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...","Thu Aug 06, 2020 04:01 UTC",Long March 2D | Gaofen-9 04 & Q-SAT,StatusActive,29.75,Success
2,2,2,SpaceX,"Pad A, Boca Chica, Texas, USA","Tue Aug 04, 2020 23:57 UTC",Starship Prototype | 150 Meter Hop,StatusActive,NaN,Success
3,3,3,Roscosmos,"Site 200/39, Baikonur Cosmodrome, Kazakhstan","Thu Jul 30, 2020 21:25 UTC",Proton-M/Briz-M | Ekspress-80 & Ekspress-103,StatusActive,65.0,Success
4,4,4,ULA,"SLC-41, Cape Canaveral AFS, Florida, USA","Thu Jul 30, 2020 11:50 UTC",Atlas V 541 | Perseverance,StatusActive,145.0,Success


# Preliminary Data Exploration

In [100]:
print(f"Shape of the dataset: {df_data.shape}")
print(f"Number of rows: {df_data.shape[0]}")
print(f"Number of columns: {df_data.shape[1]}")


Shape of the dataset: (4324, 9)
Number of rows: 4324
Number of columns: 9


In [101]:
print(f"Column names: {df_data.columns.tolist()}")

Column names: ['Unnamed: 0.1', 'Unnamed: 0', 'Organisation', 'Location', 'Date', 'Detail', 'Rocket_Status', 'Price', 'Mission_Status']


## Data Cleaning

In [102]:
df_data.isna().values.any()

np.True_

In [103]:
print(df_data.isna().sum())

Unnamed: 0.1         0
Unnamed: 0           0
Organisation         0
Location             0
Date                 0
Detail               0
Rocket_Status        0
Price             3360
Mission_Status       0
dtype: int64


In [104]:
duplicates = df_data.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

Number of duplicate rows: 0


In [105]:
df_data = df_data.drop_duplicates()

In [106]:
df_data.drop(['Unnamed: 0', 'Unnamed: 0.1'], axis=1, inplace=True, errors='ignore')

In [107]:
print(f"Column names: {df_data.columns.tolist()}")

Column names: ['Organisation', 'Location', 'Date', 'Detail', 'Rocket_Status', 'Price', 'Mission_Status']


## Descriptive Statistics

In [108]:
df_data.describe()

,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status
count,4324,4324,4324,4324,4324,964,4324
unique,56,137,4319,4278,2,56,4
top,RVSN USSR,"Site 31/6, Baikonur Cosmodrome, Kazakhstan","Tue Aug 28, 1990 09:05 UTC",Cosmos-3MRB (65MRB) | BOR-5 Shuttle,StatusRetired,450.0,Success
freq,1777,235,2,6,3534,136,3879


In [109]:
df_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4324 entries, 0 to 4323
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Organisation    4324 non-null   object
 1   Location        4324 non-null   object
 2   Date            4324 non-null   object
 3   Detail          4324 non-null   object
 4   Rocket_Status   4324 non-null   object
 5   Price           964 non-null    object
 6   Mission_Status  4324 non-null   object
dtypes: object(7)
memory usage: 236.6+ KB


In [110]:
df_data.head()

,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status
0,SpaceX,"LC-39A, Kennedy Space Center, Florida, USA","Fri Aug 07, 2020 05:12 UTC",Falcon 9 Block 5 | Starlink V1 L9 & BlackSky,StatusActive,50.0,Success
1,CASC,"Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...","Thu Aug 06, 2020 04:01 UTC",Long March 2D | Gaofen-9 04 & Q-SAT,StatusActive,29.75,Success
2,SpaceX,"Pad A, Boca Chica, Texas, USA","Tue Aug 04, 2020 23:57 UTC",Starship Prototype | 150 Meter Hop,StatusActive,NaN,Success
3,Roscosmos,"Site 200/39, Baikonur Cosmodrome, Kazakhstan","Thu Jul 30, 2020 21:25 UTC",Proton-M/Briz-M | Ekspress-80 & Ekspress-103,StatusActive,65.0,Success
4,ULA,"SLC-41, Cape Canaveral AFS, Florida, USA","Thu Jul 30, 2020 11:50 UTC",Atlas V 541 | Perseverance,StatusActive,145.0,Success


# Number of Launches per Company

In [111]:
import plotly.express as px
org_launches = df_data['Organisation'].value_counts().reset_index()
org_launches.columns = ['Organisation', 'Count']

In [112]:
# Create the Horizontal
fig = px.bar(org_launches.head(30),
             x='Count',
             y='Organisation',
             orientation='h',
             color='Count',
             color_continuous_scale='Viridis',
             title='Total Launches by Organisation (Top 30)')

# Sort by total so the leader is at the top
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

# Number of Active versus Retired Rockets

In [113]:
import plotly.express as px
# 1. Count the occurrences of each status
status_counts = df_data['Rocket_Status'].value_counts()

In [114]:
# 2. Create a Pie or Donut Chart
fig = px.pie(names=status_counts.index,
             values=status_counts.values,
             title='Rocket Status: Active vs. Retired',
             hole=0.4,
             color_discrete_sequence=['#2c3e50', '#e74c3c'])

fig.update_traces(textinfo='percent+label')
fig.show()

# Distribution of Mission Status

In [115]:
df_data['Mission_Status'].value_counts()

,count
Mission_Status,
Success,3879
Failure,339
Partial Failure,102
Prelaunch Failure,4


In [116]:
status_counts = df_data['Mission_Status'].value_counts().reset_index()
status_counts.columns = ['Status', 'Count']

In [117]:
# 2. Create a Bar Chart
fig = px.bar(status_counts,
             x='Status',
             y='Count',
             color='Status',
             color_discrete_map={
                 'Success': '#27ae60',
                 'Failure': '#c0392b',
                 'Partial Failure': '#f39c12',
                 'Prelaunch Failure': '#7f8c8d'
             },
             title='Global Mission Status Distribution (1957 - Present)')

fig.update_layout(xaxis_title='Mission Outcome', yaxis_title='Number of Missions')
fig.show()

#  A histogram and visualise the distribution.

In [118]:
# 1. Clean the Price column
df_data['Price'] = pd.to_numeric(df_data['Price'], errors='coerce')

# 2. Filter out the missing values
df_price_clean = df_data.dropna(subset=['Price'])

# 3. Create the Histogram
fig = px.histogram(df_price_clean,
                   x="Price",
                   nbins=40,
                   title="Distribution of Space Mission Launch Costs",
                   labels={'Price': 'Launch Cost (USD Millions)'},
                   color_discrete_sequence=['#3498db'])

fig.update_layout(xaxis_title='Price in Millions (USD)', yaxis_title='Count of Missions')
fig.show()

# Choropleth Map to Show the Number of Launches by Country

In [119]:
import plotly.express as px
from iso3166 import countries

# 1. Extract the potential country name (usually at the end)
df_data['Country'] = df_data['Location'].str.split(', ').str[-1]

# 2. Manual Wrangling of Anomalies
# We create a dictionary to map the "Scraped" names to "Official" names
clean_map = {
    'Russia': 'Russian Federation',
    'New Mexico': 'United States',
    'Yellow Sea': 'China',
    'Shahrud Missile Test Site': 'Iran, Islamic Republic of',
    'Pacific Missile Range Facility': 'United States',
    'Barents Sea': 'Russian Federation',
    'Gran Canaria': 'United States',
    'USA': 'United States',
    'Iran': 'Iran, Islamic Republic of',
    'South Korea': 'Korea, Republic of',
    'North Korea': "Korea, Democratic People's Republic of"
}

df_data['Country'] = df_data['Country'].replace(clean_map)

# 3. Convert Names to ISO Alpha-3 (3-letter codes)
def get_iso(name):
    try:
        return countries.get(name).alpha3
    except:
        return None

df_data['ISO'] = df_data['Country'].apply(get_iso)

In [120]:
# Group missions by ISO code
map_data = df_data.groupby('ISO').size().reset_index(name='Launch_Count')

fig = px.choropleth(map_data,
                    locations="ISO",
                    color="Launch_Count",
                    hover_name="ISO",
                    color_continuous_scale="matter",
                    title='Total Space Launches by Country (1957-Present)')

fig.show()

In [121]:
df_data.tail()

,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status,Country,ISO
4319,US Navy,"LC-18A, Cape Canaveral AFS, Florida, USA","Wed Feb 05, 1958 07:33 UTC",Vanguard | Vanguard TV3BU,StatusRetired,NaN,Failure,United States,None
4320,AMBA,"LC-26A, Cape Canaveral AFS, Florida, USA","Sat Feb 01, 1958 03:48 UTC",Juno I | Explorer 1,StatusRetired,NaN,Success,United States,None
4321,US Navy,"LC-18A, Cape Canaveral AFS, Florida, USA","Fri Dec 06, 1957 16:44 UTC",Vanguard | Vanguard TV3,StatusRetired,NaN,Failure,United States,None
4322,RVSN USSR,"Site 1/5, Baikonur Cosmodrome, Kazakhstan","Sun Nov 03, 1957 02:30 UTC",Sputnik 8K71PS | Sputnik-2,StatusRetired,NaN,Success,Kazakhstan,KAZ
4323,RVSN USSR,"Site 1/5, Baikonur Cosmodrome, Kazakhstan","Fri Oct 04, 1957 19:28 UTC",Sputnik 8K71PS | Sputnik-1,StatusRetired,NaN,Success,Kazakhstan,KAZ


# Choropleth Map to Show the Number of Failures by Country


In [122]:
# 1. Filter the data for non-successes
failures_df = df_data[df_data['Mission_Status'] != 'Success']

# 2. Group by the ISO code we created earlier
fail_counts = failures_df.groupby('ISO').size().reset_index(name='Fail_Count')

# 3. Create the Map
fig = px.choropleth(fail_counts,
                    locations="ISO",
                    color="Fail_Count",
                    hover_name="ISO",
                    color_continuous_scale=px.colors.sequential.Reds,
                    title='Total Mission Failures by Country (1957-Present)')

fig.show()

# Plotly Sunburst Chart of the countries, organisations, and mission status.

In [123]:
fig = px.sunburst(df_data,
                  path=['Country', 'Organisation', 'Mission_Status'],
                  title='Hierarchy of Missions: Country, Organisation, and Status',
                  color='Mission_Status',
                  color_discrete_map={
                      'Success': '#27ae60',
                      'Failure': '#c0392b',
                      'Partial Failure': '#f39c12',
                      'Prelaunch Failure': '#7f8c8d'
                  })

fig.update_layout(margin=dict(t=40, l=0, r=0, b=0))
fig.show()

# Analyse the Total Amount of Money Spent by Organisation on Space Missions

In [124]:
df_money = df_data.dropna(subset=['Price']).copy()
df_money['Price'] = pd.to_numeric(df_money['Price'], errors='coerce')

# Group by Organisation and Sum
total_spent = df_money.groupby('Organisation')['Price'].sum().reset_index()

# Sort the data for a professional look
total_spent = total_spent.sort_values(by='Price', ascending=False)

# Create the Chart
fig = px.bar(total_spent.head(15),
             x='Price',
             y='Organisation',
             orientation='h',
             color='Price',
             color_continuous_scale='Sunsetdark',
             title='Total Investment in Space Missions by Organisation (USD Millions)',
             labels={'Price': 'Total Spend (Millions)', 'Organisation': 'Company'})

fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

# Analyse the Amount of Money Spent by Organisation per Launch

In [125]:
import plotly.express as px

# 1. Prepare the data: Filter for valid prices and ensure they are numeric
df_avg_price = df_data.dropna(subset=['Price']).copy()
df_avg_price['Price'] = pd.to_numeric(df_avg_price['Price'], errors='coerce')

# 2. Group by Organisation and calculate the Average (Mean)
avg_price_per_org = df_avg_price.groupby('Organisation')['Price'].mean().reset_index()

# 3. Sort by Price so the most expensive are at the top
avg_price_per_org = avg_price_per_org.sort_values(by='Price', ascending=False)

# 4. Create the Visualization
fig = px.bar(avg_price_per_org.head(20),
             x='Price',
             y='Organisation',
             orientation='h',
             color='Price',
             color_continuous_scale='Bluered',
             title='Average Cost per Mission by Organisation (USD Millions)',
             labels={'Price': 'Average Launch Price (Millions)', 'Organisation': 'Company'})

fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

# Chart the Number of Launches per Year

In [126]:
df_data['Date'] = pd.to_datetime(df_data['Date'], format='mixed', utc=True)
df_data['Year'] = df_data['Date'].dt.year
df_data['Month'] = df_data['Date'].dt.month

In [127]:
# . Group by Year and Count the number of rows
launches_per_year = df_data.groupby('Year').size().reset_index(name='Launch_Count')

# 3. Create a Line Chart
fig = px.line(launches_per_year,
              x='Year',
              y='Launch_Count',
              title='Total Number of Space Launches per Year (1957 - 2020)',
              markers=True)

fig.update_layout(xaxis_title='Year', yaxis_title='Number of Missions')
fig.show()

# Chart the Number of Launches Month-on-Month until the Present

In [128]:
# 1. Convert Date to datetime (using 'mixed' to avoid the format error)
df_data['Date'] = pd.to_datetime(df_data['Date'], format='mixed', utc=True)

# 2. Create a Month-Year column
df_data['MonthYear'] = df_data['Date'].dt.to_period('M').dt.to_timestamp()

# 3. Group by the specific month and year
mom_launches = df_data.groupby('MonthYear').size().reset_index(name='Launches')

# 4. Calculate a 12-Month Rolling Average
mom_launches['Rolling_Avg'] = mom_launches['Launches'].rolling(window=12).mean()

# 5. Identify the Peak Month
peak_row = mom_launches.loc[mom_launches['Launches'].idxmax()]
peak_date = peak_row['MonthYear'].strftime('%B %Y')
peak_val = peak_row['Launches']

# 6. Create the Visualization
fig = px.line(mom_launches, x='MonthYear', y='Launches',
              title=f'Month-on-Month Launches (Peak: {peak_date} with {peak_val} Launches)',
              labels={'MonthYear': 'Date', 'Launches': 'Number of Launches'})

# Superimpose the Rolling Average line
fig.add_scatter(x=mom_launches['MonthYear'], y=mom_launches['Rolling_Avg'],
                name='12-Month Rolling Avg', line=dict(color='red', width=2))

fig.show()

/tmp/ipykernel_3952/104578135.py:5: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_data['MonthYear'] = df_data['Date'].dt.to_period('M').dt.to_timestamp()


# Analyze the months are most popular and least popular for launches

In [129]:
# 1. Extract the month number (1-12)
df_data['Month'] = df_data['Date'].dt.month

# 2. Count launches per month
monthly_stats = df_data.groupby('Month').size().reset_index(name='Count')

# 3. Map Month numbers to Names for the chart labels
month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
               7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
monthly_stats['Month_Name'] = monthly_stats['Month'].map(month_names)

# 4. Create the Bar Chart
fig = px.bar(monthly_stats,
             x='Month_Name',
             y='Count',
             color='Count',
             color_continuous_scale='Viridis',
             title='Global Launch Popularity by Month (Seasonality Analysis)')

fig.update_layout(xaxis_title='Month', yaxis_title='Number of Missions')
fig.show()

# Line Chart to show average Launch Price varied Over Time

In [130]:
import plotly.express as px

# 1. Clean the Data
df_data['Date'] = pd.to_datetime(df_data['Date'], format='mixed', utc=True)
df_data['Year'] = df_data['Date'].dt.year

# Convert Price to numeric (removing commas if they exist)
if df_data['Price'].dtype == 'O':
    df_data['Price'] = df_data['Price'].str.replace(',', '')
df_data['Price'] = pd.to_numeric(df_data['Price'], errors='coerce')

# 2. Filter and Aggregate
avg_price_year = df_data.dropna(subset=['Price']).groupby('Year')['Price'].mean().reset_index()

# 3. Create the Line Chart
fig = px.line(avg_price_year,
              x='Year',
              y='Price',
              title='Average Launch Price Variation Over Time (USD Millions)',
              markers=True,
              line_shape='linear',
              render_mode='svg')

fig.update_layout(xaxis_title='Year', yaxis_title='Average Price (Millions USD)')
fig.show()

# Chart the Number of Launches over Time by the Top 10 Organisations.

In [131]:
import plotly.express as px

# 1. Identify the Top 10 Organisations by total mission count
top_10_names = df_data['Organisation'].value_counts().head(10).index

# 2. Filter the main dataframe to include only these organisations
df_top_10 = df_data[df_data['Organisation'].isin(top_10_names)].copy()

# 3. Ensure Date is processed and Year is extracted
df_top_10['Date'] = pd.to_datetime(df_top_10['Date'], format='mixed', utc=True)
df_top_10['Year'] = df_top_10['Date'].dt.year

# 4. Group by Year and Organisation to get the annual counts
org_year_counts = df_top_10.groupby(['Year', 'Organisation']).size().reset_index(name='Launch_Count')

# 5. Create a Multi-Line Chart
fig = px.line(org_year_counts,
              x='Year',
              y='Launch_Count',
              color='Organisation',
              title='The Shift in Dominance: Top 10 Organisations Over Time',
              markers=True)

fig.update_layout(xaxis_title='Year',
                  yaxis_title='Annual Number of Launches',
                  legend_title='Organisation')
fig.show()

# Cold War Space Race: USA vs USSR

In [132]:
import plotly.express as px

# 1. Filter for the Cold War era
cold_war_df = df_data[df_data['Year'] <= 1991].copy()

# 2. Map the organisations to their respective Superpower
def map_superpower(org):
    ussr_orgs = ['RVSN USSR', 'VKS RF', 'Kosmotras', 'OKB-586', 'Strategic Rocket Forces']
    usa_orgs = ['NASA', 'US Air Force', 'General Dynamics', 'Lockheed', 'Martin Marietta', 'Boeing', 'US Navy', 'AMBA']

    if org in ussr_orgs: return 'USSR'
    if org in usa_orgs: return 'USA'
    return 'Other'

cold_war_df['Superpower'] = cold_war_df['Organisation'].apply(map_superpower)

# 3. Filter for only the two giants
race_df = cold_war_df[cold_war_df['Superpower'] != 'Other']

# 4. Count yearly launches per superpower
race_counts = race_df.groupby(['Year', 'Superpower']).size().reset_index(name='Yearly_Launches')

# 5. Calculate Cumulative Launches
race_counts['Total_Launches'] = race_counts.groupby('Superpower')['Yearly_Launches'].cumsum()

fig = px.line(race_counts, x='Year', y='Total_Launches', color='Superpower',
              title='The Space Race: Cumulative Launches (USA vs USSR) until 1991',
              color_discrete_map={'USA': '#1f77b4', 'USSR': '#d62728'})

fig.show()

In [133]:
# Group by Superpower and Mission Status
reliability = race_df.groupby(['Superpower', 'Mission_Status']).size().reset_index(name='Count')

fig = px.bar(reliability, x='Superpower', y='Count', color='Mission_Status',
             title='Cold War Reliability: Success vs Failure (1957-1991)',
             barmode='group')
fig.show()

## Plotly Pie Chart comparing the total number of launches of the USSR and the USA

In [134]:
import plotly.express as px

# 1. Filter the dataset for the Cold War era (1957 - 1991)
df_cold_war = df_data[df_data['Year'] <= 1991].copy()

# 2. Create a categorization function to group the territories
def group_superpower(country):
    if country in ['USA', 'United States']:
        return 'USA'
    elif country in ['Kazakhstan', 'Russia', 'Russian Federation']:
        return 'USSR'
    else:
        return 'Other'

# 3. Apply the function to create a new column
df_cold_war['Superpower'] = df_cold_war['Country'].apply(group_superpower)

# 4. Filter out 'Other' countries to only compare the Big Two
df_superpowers = df_cold_war[df_cold_war['Superpower'] != 'Other']

# 5. Count the total launches for each superpower
superpower_counts = df_superpowers['Superpower'].value_counts().reset_index()
superpower_counts.columns = ['Superpower', 'Count']

# 6. Create the Pie (Donut) Chart
fig = px.pie(superpower_counts,
             names='Superpower',
             values='Count',
             title='USA vs USSR: Total Space Launches (1957 - 1991)',
             color='Superpower',
             color_discrete_map={'USA': '#1f77b4', 'USSR': '#d62728'}, # Blue vs Red
             hole=0.4) # A donut chart looks more modern!

# Show actual numbers alongside percentages
fig.update_traces(textinfo='label+percent+value', textposition='inside')
fig.show()

## Total Number of Launches Year-On-Year by the Two Superpowers

In [135]:
import plotly.express as px

# 1. Define the function to group the countries
def group_superpower(country):
    if country in ['USA', 'United States']:
        return 'USA'
    elif country in ['Kazakhstan', 'Russia', 'Russian Federation']:
        return 'USSR'
    else:
        return 'Other'

# 2. Apply the function to create the 'Superpower' column on the MAIN dataframe
df_data['Superpower'] = df_data['Country'].apply(group_superpower)

# 3. Filter out 'Other' to only compare the Big Two
df_superpowers = df_data[df_data['Superpower'] != 'Other']

# 4. Group by Year and Superpower to get the annual counts
yoy_superpowers = df_superpowers.groupby(['Year', 'Superpower']).size().reset_index(name='Launches')

# 5. Create the Area Chart
fig = px.area(yoy_superpowers,
              x='Year',
              y='Launches',
              color='Superpower',
              title='Total Global Launches Year-on-Year (USA vs USSR Contribution)',
              color_discrete_map={'USA': '#1f77b4', 'USSR': '#d62728'})

fig.update_layout(xaxis_title='Year', yaxis_title='Total Number of Launches')
fig.show()

## Chart the Total Number of Mission Failures Year on Year.

In [136]:
import plotly.express as px

# 1. Filter out the successful missions
df_failures = df_data[df_data['Mission_Status'] != 'Success']

# 2. Group by Year and the specific type of Mission Status
failures_yoy = df_failures.groupby(['Year', 'Mission_Status']).size().reset_index(name='Count')

# 3. Create a Stacked Bar Chart
fig = px.bar(failures_yoy,
             x='Year',
             y='Count',
             color='Mission_Status',
             title='Total Mission Failures Year-on-Year (1957 - Present)',
             color_discrete_map={
                 'Failure': '#c0392b',
                 'Partial Failure': '#f39c12',
                 'Prelaunch Failure': '#7f8c8d'
             })
fig.update_layout(xaxis_title='Year',
                  yaxis_title='Number of Failures',
                  barmode='stack')
fig.show()

## Chart the Percentage of Failures over Time

In [137]:
total_per_year = df_data.groupby('Year').size().reset_index(name='Total_Launches')
df_failures = df_data[df_data['Mission_Status'] != 'Success']
failures_per_year = df_failures.groupby('Year').size().reset_index(name='Failures')
failure_rate_df = pd.merge(total_per_year, failures_per_year, on='Year', how='left')
failure_rate_df['Failures'] = failure_rate_df['Failures'].fillna(0)
failure_rate_df['Failure_Percentage'] = (failure_rate_df['Failures'] / failure_rate_df['Total_Launches']) * 10
failure_rate_df['Rolling_Avg'] = failure_rate_df['Failure_Percentage'].rolling(window=5).mean()

# 6. Create the Line Chart
fig = px.line(failure_rate_df,
              x='Year',
              y='Failure_Percentage',
              title='Percentage of Mission Failures Over Time (Risk Analysis)',
              markers=True,
              labels={'Failure_Percentage': 'Failure Rate (%)'})

fig.add_scatter(x=failure_rate_df['Year'], y=failure_rate_df['Rolling_Avg'],
                name='5-Year Rolling Avg', line=dict(color='red', width=3))

fig.show()

# For Every Year Show which Country was in the Lead in terms of Total Number of Launches up to and including including 2020

In [138]:
yearly_totals = df_data[df_data['Year'] <= 2020].groupby(['Year', 'Country']).size().reset_index(name='Total')
idx_max_total = yearly_totals.groupby('Year')['Total'].idxmax()
total_leaders = yearly_totals.loc[idx_max_total]
success_df = df_data[(df_data['Mission_Status'] == 'Success') & (df_data['Year'] <= 2020)]
yearly_success = success_df.groupby(['Year', 'Country']).size().reset_index(name='Success_Count')
idx_max_success = yearly_success.groupby('Year')['Success_Count'].idxmax()
success_leaders = yearly_success.loc[idx_max_success]

fig = px.bar(total_leaders,
             x='Year',
             y='Total',
             color='Country',
             title='Global Leader in Total Launches per Year (up to 2020)',
             labels={'Total': 'Number of Launches'},
             color_discrete_sequence=px.colors.qualitative.Bold)

fig.show()

#Year-on-Year Chart Showing the Organisation Doing the Most Number of Launches

In [139]:
import plotly.express as px

# 1. Group by Year and Organisation, then count the number of launches
org_yearly_counts = df_data.groupby(['Year', 'Organisation']).size().reset_index(name='Launches')

# 2. Find the organization with the MAXIMUM launches for each year
idx_max_org = org_yearly_counts.groupby('Year')['Launches'].idxmax()
top_orgs_per_year = org_yearly_counts.loc[idx_max_org]

# 3. Create a Bar Chart
fig = px.bar(top_orgs_per_year,
             x='Year',
             y='Launches',
             color='Organisation',
             title='Global Dominance: Top Organisation by Number of Launches Each Year',
             labels={'Launches': 'Number of Launches (Leader only)'},
             color_discrete_sequence=px.colors.qualitative.Alphabet)

fig.update_layout(xaxis_title='Year', yaxis_title='Launches by the Year\'s Leader')
fig.show()